In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer.BufferTrainer import BufferTrainer
from src.trainer.SimpleTrainer import SimpleTrainer
from src.data_utils import (
    get_mnist_tasks,
    _extract_targets,
    get_context_sets,
    create_holdout_set,
)
from src.utils import InContextHead
from src import models
from src.buffer import MultiTaskBuffer

### Task Incremental Training

In [4]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cpu")
head.set_context(0)
model = models.get_fully_connected_mnist_model(head=head)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [5]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|████████████████████████████████████████| 3/3 [00:23<00:00,  7.70s/it, val_loss=0.0283, val_acc=0.993]


Test Results: [(0.02, 0.9965)]


[(0.02, 0.9965)]

In [ ]:
buffer = MultiTaskBuffer([])
interval_trainer = BufferTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.9,
    primal_learning_rate=0.5,
    paradigm="TIL",
    buffer=buffer,
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0], context_id=0)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:])
):
    interval_trainer.train(
        train, val, batch_size=256, context_id=i + 1, target_acc=0.9, patience=50
    )
    interval_trainer.test(test_tasks[0 : i + 2], context_list=list(range(i + 2)))
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test, context_id=i + 1)

---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.0960 (Positive = violated)
Number of model parameters: 101770
Computing Rashomon set with min acc limit: 0.90
Initial bbox:  Obj=0.20,  Size=20640.00,  Min acc hard=1.00,  Min acc soft=1.00


100%|███████████████████████████████████████| 300/300 [00:19<00:00, 15.65it/s, size=24648.85, obj=0.242, min_soft_acc=0.877]


Final bbox:  Obj=0.24,  Size=24648.85,  Min acc hard=0.90,  Min acc soft=0.90
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 15 checkpoints
Checkpoints sizes: ['20696.63', '20997.11', '21348.07', '21678.77', '21983.71', '22276.16', '22543.12', '22819.94', '23084.97', '23358.41', '23619.99', '23878.26', '24139.37', '24394.04', '24648.85']
Checkpoint certificates: ['0.99', '0.96', '0.92', '0.89', '0.90', '0.90', '0.91', '0.89', '0.90', '0.90', '0.90', '0.90', '0.90', '0.90', '0.90']
----------------------- Finished Computing Rashomon set ------------------------


Processing items: 5it [00:04,  1.02it/s, Training loss=0.226, Training accuracy=0.93, Buffer Data Consumed=0] 


Exiting with final training accuracy of 0.9297
Test Results: [(0.0192, 0.996), (0.203, 0.9395)]
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.0207 (Positive = violated)
Computing Rashomon set within outer box of size: 22276.16
Number of model parameters: 101770
Computing Rashomon set with min acc limit: 0.90
Initial bbox:  Obj=0.20,  Size=20640.00,  Min acc hard=0.92,  Min acc soft=0.92


...

Exiting with final training accuracy of 0.9258
Test Results: [(0.0192, 0.9955), (0.2233, 0.937), (0.1761, 0.9864), (0.3235, 0.9338)]
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.0305 (Positive = violated)
Computing Rashomon set within outer box of size: 11622.06
Number of model parameters: 101770
Computing Rashomon set with min acc limit: 0.89
Initial bbox:  Obj=0.20,  Size=20640.00,  Min acc hard=0.92,  Min acc soft=0.92


100%|████████████████████████████████████████| 300/300 [00:18<00:00, 16.33it/s, size=6377.14, obj=0.063, min_soft_acc=0.867]


Final bbox:  Obj=0.06,  Size=6377.14,  Min acc hard=0.88,  Min acc soft=0.87
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 15 checkpoints
Checkpoints sizes: ['5196.62', '5360.18', '5568.57', '5772.37', '5963.03', '6142.11', '6194.54', '6246.32', '6294.15', '6337.27', '6372.71', '6375.26', '6376.54', '6376.75', '6377.14']
Checkpoint certificates: ['0.88', '0.88', '0.88', '0.88', '0.87', '0.87', '0.88', '0.88', '0.88', '0.88', '0.88', '0.88', '0.88', '0.88', '0.88']
----------------------- Finished Computing Rashomon set ------------------------


Processing items: 16it [00:18,  1.17s/it, Training loss=0.215, Training accuracy=0.969, Buffer Data Consumed=0]


Exiting with final training accuracy of 0.9688
Test Results: [(0.0191, 0.9955), (0.2244, 0.9365), (0.1777, 0.9844), (0.3358, 0.93), (0.2213, 0.9728)]


### Domain Incremental Training

In [7]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=0)

model = models.get_mnist_model(seed=0)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


In [8]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [9]:
train_tasks, buffer_sets = zip(
    *[create_holdout_set(dataset) for dataset in train_tasks]
)
print([len(task) for task in buffer_sets])

[965, 951, 1014, 963, 904]


In [ ]:
trainer = SimpleTrainer(model, domain_map_fn=domain_map_fn)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=64)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|████████████████████████████████████████| 3/3 [01:11<00:00, 23.72s/it, val_loss=0.0167, val_acc=0.995]


Test Results: [(0.0308, 0.9875)]


[(0.0308, 0.9875)]

In [11]:
buffer = MultiTaskBuffer([])
interval_trainer = BufferTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=20,
    min_acc_limit=0.9,
    primal_learning_rate=0.5,
    paradigm="DIL",
    buffer=buffer,
    domain_map_fn=domain_map_fn,
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])
interval_trainer.add_to_buffer(buffer_sets[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, target_acc=0.9)
    interval_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test)
        interval_trainer.add_to_buffer(buffer_sets[i])

---------------------------- Computing Rashomon set ----------------------------


Processing items: 4it [00:03,  1.22it/s, Training loss=0.117, Training accuracy=0.984, Buffer Data Consumed=0] 


Exiting with final training accuracy of 0.9844
Test Results: [(0.9861, 0.5504), (0.1074, 0.9809)]
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.0666 (Positive = violated)
Computing Rashomon set within outer box of size: 123580.00
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.90
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.97,  Min acc soft=0.97


100%|████████████████████████████████████████████| 20/20 [00:02<00:00,  7.73it/s, size=43.99, obj=0.006, min_soft_acc=0.962]


Final bbox:  Obj=0.01,  Size=43.99,  Min acc hard=0.95,  Min acc soft=0.95
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 1 checkpoints
Checkpoints sizes: ['43.99']
Checkpoint certificates: ['0.95']
----------------------- Finished Computing Rashomon set ------------------------


...

Final bbox:  Obj=0.00,  Size=29.75,  Min acc hard=0.85,  Min acc soft=0.84
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 1 checkpoints
Checkpoints sizes: ['29.75']
Checkpoint certificates: ['0.85']
----------------------- Finished Computing Rashomon set ------------------------


Processing items: 1it [00:00,  2.79it/s, Training loss=1.03, Training accuracy=0.492, Buffer Data Consumed=1600] 


Exiting with final training accuracy of 0.4922
Test Results: [(0.7495, 0.6144), (0.1195, 0.9744), (0.3574, 0.8722), (0.9414, 0.5376)]


Processing items: 1it [00:00,  1.34it/s, Training loss=1.54, Training accuracy=0.23, Buffer Data Consumed=1600] 


Exiting with final training accuracy of 0.2305
Test Results: [(0.7182, 0.6139), (0.1416, 0.9658), (0.3513, 0.8814), (0.9907, 0.5208), (1.4697, 0.2343)]


### Class Incremental Learning

In [12]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [13]:
train_tasks, buffer_sets = zip(
    *[create_holdout_set(dataset) for dataset in train_tasks]
)
print([len(task) for task in buffer_sets])

[956, 945, 973, 910, 1014]


In [14]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|█████████████████████████████████████████| 3/3 [00:20<00:00,  6.97s/it, val_loss=0.0371, val_acc=0.99]

Test Results: [(0.0218, 0.9925)]


[(0.0218, 0.9925)]

In [15]:
buffer = MultiTaskBuffer([])
interval_trainer = BufferTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=20,
    min_acc_limit=0.8,
    primal_learning_rate=0.5,
    paradigm="CIL",
    buffer=buffer,
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])
interval_trainer.add_to_buffer(buffer_sets[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, target_acc=0.9)
    interval_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test)
        interval_trainer.add_to_buffer(buffer_sets[i])

---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1883 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|████████████████████████████████████████████| 20/20 [00:02<00:00,  7.41it/s, size=41.26, obj=0.006, min_soft_acc=0.940]


Final bbox:  Obj=0.01,  Size=41.26,  Min acc hard=0.95,  Min acc soft=0.95
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 1 checkpoints
Checkpoints sizes: ['41.26']
Checkpoint certificates: ['0.95']
----------------------- Finished Computing Rashomon set ------------------------


Processing items: 1it [00:00,  2.35it/s, Training loss=26.6, Training accuracy=0, Buffer Data Consumed=0]

Using buffer to continue training.
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1847 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.99,  Min acc soft=0.98


...

Exiting with final training accuracy of 0.0000
Test Results: [(0.0332, 0.991), (16.3762, 0.0), (23.6878, 0.0)]
---------------------------- Computing Rashomon set ----------------------------


Processing items: 1it [00:00,  1.35it/s, Training loss=25.7, Training accuracy=0, Buffer Data Consumed=800]


Exiting with final training accuracy of 0.0000
Test Results: [(0.0332, 0.991), (16.3762, 0.0), (27.1148, 0.0), (26.6456, 0.0)]


Processing items: 1it [00:00,  1.54it/s, Training loss=21.1, Training accuracy=0, Buffer Data Consumed=800]


Exiting with final training accuracy of 0.0000
Test Results: [(0.0332, 0.991), (16.3762, 0.0), (27.1311, 0.0), (30.6734, 0.0), (21.0783, 0.0)]
